In [4]:
# Analyze new long-format claims and raw responses
import pandas as pd
from pathlib import Path

# Inputs derived from process_batch outputs
base1 = 'results_80B'  # adjust if different base names
base2 = 'results'      # adjust if different base names
raw1 = Path(f"data/datasets/{base1}.csv")
raw2 = Path(f"data/datasets/{base2}.csv")
claims1 = Path(f"{base1}_claims.csv")
claims2 = Path(f"{base2}_claims.csv")

# Helper to load if exists

def load_csv(path: Path) -> pd.DataFrame | None:
    if path.exists():
        try:
            return pd.read_csv(path)
        except Exception as e:
            print(f"Failed to read {path}: {e}")
    else:
        print(f"Warning: {path} not found")
    return None

raw_df_80B = load_csv(raw1)
raw_df_base = load_csv(raw2)
claims_df_80B = load_csv(claims1)
claims_df_base = load_csv(claims2)

# Summary: raw responses
for label, rdf in [('80B', raw_df_80B), ('base', raw_df_base)]:
    if rdf is None:
        continue
    total = len(rdf)
    empty = int(rdf['raw_response'].fillna('').eq('').sum()) if 'raw_response' in rdf.columns else 0
    print(f"Raw {label}: rows={total}, empty_raw={empty}")

# Summary: claims long format
for label, cdf in [('80B', claims_df_80B), ('base', claims_df_base)]:
    if cdf is None:
        continue
    total = len(cdf)
    non_empty = int(cdf['claim'].fillna('').ne('').sum()) if 'claim' in cdf.columns else 0
    unique_cases = cdf['name'].nunique() if 'name' in cdf.columns else None
    print(f"Claims {label}: rows={total}, non_empty_claims={non_empty}, unique_cases={unique_cases}")

# Show examples: first 5 claims for Supreme Court cases

def is_supreme_case(row):
    court = str(row.get('court', '')).lower()
    name = str(row.get('name', '')).lower()
    docket = str(row.get('docket', '')).lower()
    return ('supreme' in court) or ('supreme court' in name) or ('scotus' in court) or ('supreme' in docket)

for label, cdf in [('80B', claims_df_80B), ('base', claims_df_base)]:
    if cdf is None:
        continue
    # If claims csv lacks court, we can’t filter by court; fallback to first 5
    subset = cdf
    if {'court','name','docket'}.issubset(set(cdf.columns)):
        subset = cdf[cdf.apply(is_supreme_case, axis=1)]
    sample = subset.head(5)
    print(f"\n{label}: first 5 claims (Supreme Court filtered when possible)")
    display(sample.fillna(''))


Raw 80B: rows=3299, empty_raw=0
Raw base: rows=3299, empty_raw=0
Claims 80B: rows=10480, non_empty_claims=10480, unique_cases=3234
Claims base: rows=10472, non_empty_claims=10472, unique_cases=3234

80B: first 5 claims (Supreme Court filtered when possible)


,name,docket,claim
0,Roe v. Wade,70-18,The Constitution protects a person's right to ...
1,Roe v. Wade,70-18,States cannot restrict abortion during the fir...
2,Roe v. Wade,70-18,"During the second three months of pregnancy, s..."
3,Roe v. Wade,70-18,"After the fetus can survive outside the womb, ..."
4,Stanley v. Illinois,70-5014,States cannot take children from unwed fathers...



base: first 5 claims (Supreme Court filtered when possible)


,name,docket,claim
0,Roe v. Wade,70-18,The Constitution protects a right to privacy t...
1,Roe v. Wade,70-18,States cannot restrict abortion during the fir...
2,Roe v. Wade,70-18,States may regulate abortion procedures during...
3,Roe v. Wade,70-18,"After the fetus can survive outside the womb, ..."
4,Stanley v. Illinois,70-5014,States cannot automatically assume unwed fathe...
